# Projet — Exploitation du dateset CL-Drive

## Estimation de la charge cognitive du conducteur

Ce TP vient après les TD1, TD2, TD3 et TD4.

Les TD ont déjà permis de travailler :

- la compréhension du papier CL-Drive ;
- le protocole expérimental ;
- la segmentation en fenêtres de 10 s ;
- le prétraitement EEG ;
- l'extraction des features EEG.

Le point de départ du TP est donc le dossier généré à la fin du TD4 :

```text
EEG_Features_10s/
```

Ce TP ne revient pas sur le calcul des features. Il exploite les features déjà extraites pour construire un pipeline d'apprentissage automatique.

## Objectif du TP

Construire un pipeline complet :

```text
EEG_Features_10s
→ Normalized_Features_10s/EEG
→ Normalized_Features_10s_With_Label/EEG
→ Dataset EEG supervisé
→ Classification de la charge cognitive
→ Évaluation
→ Interprétation
```

Dans un premier temps, on se limite à l'EEG uniquement.

La multimodalité, c'est-à-dire l'ajout de ECG, EDA et Gaze, sera proposée uniquement comme extension à la fin du sujet.

## 1. Structure attendue des dossiers

Avant de commencer, le dossier de travail doit contenir au minimum :

```text
Data/
├── EEG/ID_x
│   ├── ... fichiers level_1, level_2, ..., level_9
│   ├── ... fichiers baseline
│   └── ... fichiers filtered_*
│
├── EEG_Features_10s/
│   ├── ID1_EEG_features.csv
│   ├── ID2_EEG_features.csv
│   └── ...
│
├── Labels/
│   ├── ID1.csv
│   ├── ID2.csv
│   └── ...
```

Le TP va générer deux nouveaux dossiers :

```text
Data/
├── Normalized_Features_10s/
│   └── EEG/
│       ├── norm_ID1_EEG_features.csv
│       ├── norm_ID2_EEG_features.csv
│       └── ...
│
├── Normalized_Features_10s_With_Label/(avec colonnes Level et Label)
│   └── EEG/
│       ├── norm_ID1_EEG_features.csv
│       ├── norm_ID2_EEG_features.csv
│       └── ...
```

## Question 

Pourquoi ne faut-il pas entraîner directement les modèles sur les fichiers `EEG_Features_10s`  ?

### Réponse 

Elles ne sont pas normalisées à la baseline. Donc on risque d'apprendre les différences entre sujets.

In [5]:
from pathlib import Path
import re
import pandas as pd

# TODO : adapter ce chemin à votre organisation locale.
BASE_PATH = Path("Data")

EEG_FEATURE_DIR = BASE_PATH / "EEG_Features_10s"
LABEL_DIR = BASE_PATH / "Labels"
NORMALIZED_ROOT = BASE_PATH / "Normalized_Features_10s"
NORMALIZED_EEG_DIR = NORMALIZED_ROOT / "EEG"
LABELED_ROOT = BASE_PATH / "Normalized_Features_10s_With_Label"
LABELED_EEG_DIR = LABELED_ROOT / "EEG"

NORMALIZED_EEG_DIR.mkdir(parents=True, exist_ok=True)
LABELED_EEG_DIR.mkdir(parents=True, exist_ok=True)

METADATA_COLUMNS = ["Participant", "File", "Window", "Start_Time", "End_Time", "Channel",
                    "subject", "scenario", "window", "t_start", "t_end", "canal"]

## 2. Normalisation des features EEG

À la fin du TD4, chaque fichier CSV contient des features EEG calculées sur des fenêtres de 10 secondes.

La normalisation doit suivre deux étapes :

### Étape 1 — Normalisation par la baseline du sujet

Pour chaque sujet, les fichiers de baseline servent à calculer une valeur moyenne de référence pour chaque feature :

$$
\mu_{baseline}^{(s,f)} = \frac{1}{N}\sum_{i=1}^{N} x_i^{(s,f)}
$$

où :

- $s$ désigne le sujet ;
- $f$ désigne la feature ;
- $x_i^{(s,f)}$ désigne la valeur de la feature pendant la baseline.

Chaque valeur de feature dans les fichiers de tâche est ensuite divisée par la moyenne de baseline correspondante :

$$
x_{norm}^{(s,f)} = \frac{ x^{(s,f)} }{ \mu_{baseline}^{(s,f)} }
$$

### Étape 2 — Standardisation z-score

On applique ensuite une standardisation :

$$
z = \frac{x - \mu}{\sigma}
$$

Cela permet d’obtenir des features centrées et réduites. Cette étape devra toutefois être réalisée plus loin dans le pipeline, après la séparation des données entre les ensembles d’entraînement et de test (voir section 8 ci-dessous).

## Question

Quel est l'intérêt de la normalisation par baseline dans des signaux physiologiques ?

### Réponse 

Elle permet de comparer les variations par rapport à l'état de repos de chaque sujet. Ça réduit les différences naturelles entre participants et rend les features plus comparables.

In [6]:
def get_feature_columns(df, metadata_columns=METADATA_COLUMNS):
    """
    Retourne les colonnes numériques correspondant aux features.
    Les colonnes de métadonnées ne doivent pas être normalisées.
    """
    numeric_cols = df.select_dtypes(include="number").columns
    return [col for col in numeric_cols if col not in metadata_columns]


def get_participant_column(df):
    if "Participant" in df.columns:
        return "Participant"
    return "subject"


def get_file_column(df):
    if "File" in df.columns:
        return "File"
    return "scenario"


def compute_baseline_averages(feature_dir):
    """
    Calcule, pour chaque Participant, la moyenne de baseline de chaque feature.

    Indications :
    - parcourir les fichiers CSV de feature_dir ;
    - garder uniquement les fichiers dont le nom contient 'baseline' ;
    - lire chaque fichier avec pandas.read_csv ;
    - identifier le Participant avec df['Participant'].iloc[0] ;
    - calculer la moyenne de chaque feature ;
    """
    baseline_avgs = {}

    for file_path in Path(feature_dir).glob("*.csv"):
        df = pd.read_csv(file_path)
        file_col = get_file_column(df)

        if "baseline" in file_path.name.lower():
            df_baseline = df
        else:
            df_baseline = df[df[file_col].astype(str).str.contains("baseline", case=False, na=False)]

        if df_baseline.empty:
            continue

        participant_col = get_participant_column(df_baseline)
        participant_id = df_baseline[participant_col].iloc[0]
        feature_cols = get_feature_columns(df_baseline)

        baseline_avgs[participant_id] = df_baseline[feature_cols].mean()

    return baseline_avgs


def normalize_by_baseline(df, participant_id, baseline_avgs):
    """
    Divise chaque feature par sa moyenne de baseline pour le sujet considéré.
    """
    df_norm = df.copy()
    feature_cols = get_feature_columns(df_norm)

    if participant_id not in baseline_avgs:
        print(f"Baseline manquante pour {participant_id}")
        return df_norm

    baseline = baseline_avgs[participant_id].replace(0, pd.NA)
    df_norm[feature_cols] = df_norm[feature_cols].div(baseline[feature_cols])

    return df_norm



def run_eeg_normalization():
    """
    Génère les fichiers du dossier :
    Normalized_Features_10s/EEG
    à partir du dossier :
    EEG_Features_10s
    """
    baseline_avgs = compute_baseline_averages(EEG_FEATURE_DIR)

    for file_path in EEG_FEATURE_DIR.glob("*.csv"):
        df = pd.read_csv(file_path)
        file_col = get_file_column(df)

        if "baseline" in file_path.name.lower():
            continue

        df = df[~df[file_col].astype(str).str.contains("baseline", case=False, na=False)]
        participant_col = get_participant_column(df)
        participant_id = df[participant_col].iloc[0]

        df_norm = normalize_by_baseline(df, participant_id, baseline_avgs)
        output_file = NORMALIZED_EEG_DIR / f"norm_{file_path.name}"
        df_norm.to_csv(output_file, index=False)

        print(f"Sauvegardé : {output_file}")

In [7]:
run_eeg_normalization()

Baseline manquante pour 1030
Sauvegardé : Data\Normalized_Features_10s\EEG\norm_1030_features.csv
Baseline manquante pour 1105
Sauvegardé : Data\Normalized_Features_10s\EEG\norm_1105_features.csv
Baseline manquante pour 1106
Sauvegardé : Data\Normalized_Features_10s\EEG\norm_1106_features.csv
Baseline manquante pour 1241
Sauvegardé : Data\Normalized_Features_10s\EEG\norm_1241_features.csv
Baseline manquante pour 1271
Sauvegardé : Data\Normalized_Features_10s\EEG\norm_1271_features.csv
Baseline manquante pour 1314
Sauvegardé : Data\Normalized_Features_10s\EEG\norm_1314_features.csv
Baseline manquante pour 1323
Sauvegardé : Data\Normalized_Features_10s\EEG\norm_1323_features.csv
Baseline manquante pour 1337
Sauvegardé : Data\Normalized_Features_10s\EEG\norm_1337_features.csv
Baseline manquante pour 1372
Sauvegardé : Data\Normalized_Features_10s\EEG\norm_1372_features.csv
Baseline manquante pour 1417
Sauvegardé : Data\Normalized_Features_10s\EEG\norm_1417_features.csv
Baseline manquante p

## 3. Vérification du dossier `Normalized_Features_10s/EEG`

Après exécution de la normalisation, vérifiez que le dossier contient bien des fichiers `norm_*.csv`.

## Question

Pourquoi les fichiers de baseline ne sont-ils pas copiés dans le dossier normalisé final ?

### Réponse 

Les fichiers de baseline servent seulement de référence pour normaliser les autres fichiers. On ne les garde pas pour l'entraînement car ils ne correspondent pas aux tâches à classifier.

In [8]:
# Vérification du dossier Normalized_Features_10s/EEG
norm_files = list(NORMALIZED_EEG_DIR.glob("norm_*.csv"))

print("Nombre de fichiers normalisés :", len(norm_files))

if len(norm_files) > 0:
    print("Premier fichier :", norm_files[0].name)
    df_check = pd.read_csv(norm_files[0])
    display(df_check.head())
else:
    print("Aucun fichier norm_*.csv trouvé.")


Nombre de fichiers normalisés : 21
Premier fichier : norm_1030_features.csv


,delta_abs,delta_mean,delta_max,delta_min,delta_median,theta_abs,theta_mean,theta_max,theta_min,theta_median,...,raw_max,raw_median,raw_var,raw_std,subject,scenario,window,canal,t_start,t_end
0,2726.690058,389.527151,1224.651028,150.053743,216.815608,733.296537,91.662067,143.219328,34.293231,92.821110,...,190.917969,-26.367188,3240.390047,56.924424,1030,level_1,0,TP9,120.007812,130.003906
1,305.508778,43.644111,92.959505,13.898414,44.890653,70.205669,8.775709,10.884229,2.935216,9.812664,...,52.246094,-20.507812,303.682496,17.426488,1030,level_1,0,AF7,120.007812,130.003906
2,585.943415,83.706202,105.701601,52.037081,92.189125,186.122574,23.265322,55.207564,4.274546,19.007681,...,120.605469,-17.578125,598.824955,24.470900,1030,level_1,0,AF8,120.007812,130.003906
3,1459.946168,208.563738,318.585456,125.646682,213.451503,695.372477,86.921560,161.087055,37.776120,79.889700,...,117.675781,-18.554688,1418.860131,37.667760,1030,level_1,0,TP10,120.007812,130.003906
4,3166.055472,452.293639,554.449619,374.010425,425.324039,1488.042508,186.005313,305.065202,89.671106,197.099737,...,136.718750,-20.019531,2489.655784,49.896451,1030,level_1,1,TP9,130.007812,140.003906


## 4. Ajout des colonnes `Level` et `Label`

Les fichiers normalisés ne contiennent pas encore la cible d'apprentissage.

Il faut maintenant associer chaque fenêtre de 10 secondes à son score PAAS.

Les labels sont stockés dans le dossier :

```text
Labels/
```

Chaque fichier de labels correspond à un sujet, par exemple :

```text
Labels/ID1.csv
Labels/ID2.csv
...
```

Dans ces fichiers, on suppose une structure du type :

| time | lvl_1 | lvl_2 | ... | lvl_9 |
|---:|---:|---:|---|---:|
| 10 | 2 | 3 | ... | 5 |
| 20 | 2 | 4 | ... | 6 |
| ... | ... | ... | ... | ... |

Pour une fenêtre d'indice `Window`, le temps associé est :

$$
time = (Window + 1) \times 10
$$

Le niveau du scénario est extrait du nom du fichier avec une expression régulière :

```text
level_1 → Level = 1
level_2 → Level = 2
...
level_9 → Level = 9
```

Le score PAAS est ensuite récupéré dans la colonne :

```text
lvl_<Level>
```

Exemple : si `Level = 4`, on lit la colonne `lvl_4`.


In [11]:
import re


def extract_level_from_filename(file_name):
    """
    Extrait le niveau de scénario à partir du nom de fichier.

    Exemple :
    filtered_level_3.csv → 3
    norm_filtered_level_8.csv → 8
    """
    match = re.search(r"level[_-]?(\d+)", str(file_name))
    if match:
        return int(match.group(1))
    return None


def get_label_for_row(row, labels_df):
    """
    Retourne le score PAAS correspondant à une ligne de features.

    Indications :
    - récupérer Window ;
    - calculer time_stamp = (Window + 1) * 10 ;
    - récupérer Level ;
    - construire label_col = f'lvl_{Level}' ;
    - chercher dans labels_df la ligne où labels_df['time'] == time_stamp ;
    - retourner la valeur de label_col.
    """
    window_col = "Window" if "Window" in row.index else "window"
    time_stamp = (int(row[window_col]) + 1) * 10
    label_col = f"lvl_{int(row['Level'])}"

    match = labels_df[labels_df["time"] == time_stamp]
    if match.empty or label_col not in labels_df.columns:
        return None

    return match[label_col].iloc[0]


def attach_labels_eeg():
    """
    Génère les fichiers du dossier :
    Normalized_Features_10s_With_Label/EEG

    Chaque fichier de sortie doit contenir deux nouvelles colonnes :
    - Level
    - Label
    """
    for file_path in NORMALIZED_EEG_DIR.glob("norm_*.csv"):
        df = pd.read_csv(file_path)

        participant_col = get_participant_column(df)
        participant_id = df[participant_col].iloc[0]

        labels_path = LABEL_DIR / f"{participant_id}.csv"
        labels_df = pd.read_csv(labels_path)

        file_col = get_file_column(df)
        df["Level"] = df[file_col].apply(extract_level_from_filename)

        if df["Level"].isna().all():
            df["Level"] = extract_level_from_filename(file_path.name)

        df["Label"] = df.apply(lambda row: get_label_for_row(row, labels_df), axis=1)
        df = df.dropna(subset=["Level", "Label"])

        output_file = LABELED_EEG_DIR / file_path.name
        df.to_csv(output_file, index=False)

        print(f"Sauvegardé : {output_file}")

In [12]:
attach_labels_eeg()

Sauvegardé : Data\Normalized_Features_10s_With_Label\EEG\norm_1030_features.csv
Sauvegardé : Data\Normalized_Features_10s_With_Label\EEG\norm_1105_features.csv
Sauvegardé : Data\Normalized_Features_10s_With_Label\EEG\norm_1106_features.csv
Sauvegardé : Data\Normalized_Features_10s_With_Label\EEG\norm_1241_features.csv
Sauvegardé : Data\Normalized_Features_10s_With_Label\EEG\norm_1271_features.csv
Sauvegardé : Data\Normalized_Features_10s_With_Label\EEG\norm_1314_features.csv
Sauvegardé : Data\Normalized_Features_10s_With_Label\EEG\norm_1323_features.csv
Sauvegardé : Data\Normalized_Features_10s_With_Label\EEG\norm_1337_features.csv
Sauvegardé : Data\Normalized_Features_10s_With_Label\EEG\norm_1372_features.csv
Sauvegardé : Data\Normalized_Features_10s_With_Label\EEG\norm_1417_features.csv
Sauvegardé : Data\Normalized_Features_10s_With_Label\EEG\norm_1434_features.csv
Sauvegardé : Data\Normalized_Features_10s_With_Label\EEG\norm_1544_features.csv
Sauvegardé : Data\Normalized_Features_10

## 5. Vérification du dossier `Normalized_Features_10s_With_Label/EEG`

Le dossier final doit contenir des fichiers CSV avec au moins :

- les métadonnées : `Participant`, `File`, `Window`, `Channel`, `Start_Time` et `End_Time` ;
- les features EEG normalisées ;
- la colonne `Level` ;
- la colonne `Label`.

## Question

Quelle est la différence entre `Level` et `Label` dans ce TP ? Pourquoi faut-il ajouter à la fois `Level` et `Label` ?

### Réponse

...

In [ ]:
# Vérification du dossier Normalized_Features_10s_With_Label/EEG


## 6. Construction du dataset EEG supervisé

Une fois les fichiers normalisés et labellisés générés, on peut les concaténer pour construire un tableau unique.

Chaque ligne représente une fenêtre EEG de 10 secondes pour un canal.

On construit ensuite deux problèmes possibles :

### Classification binaire

| Score PAAS | Classe |
|---:|---|
| 1 à 4 | faible |
| 5 à 9 | élevée |

### Classification ternaire, extension

| Score PAAS | Classe |
|---:|---|
| 1 à 3 | faible |
| 4 à 6 | moyenne |
| 7 à 9 | élevée |

Dans ce TP, l'objectif principal est la classification binaire.

In [ ]:
def load_labeled_eeg_dataset():
    """
    Concatène tous les fichiers CSV du dossier Normalized_Features_10s_With_Label/EEG.
    """
    # TODO : parcourir LABELED_EEG_DIR, lire les CSV, concaténer avec pd.concat.


df = load_labeled_eeg_dataset()
print(df.shape)
df.head()

In [ ]:
# Création des cibles de classification.
df["Label_Binary"] = ...

# Extension ternaire éventuelle.
df["Label_Ternary"] =...

## 7. Préparation de la matrice d'apprentissage

On doit séparer :

- les métadonnées ;
- les features numériques EEG ;
- la cible d'apprentissage.

## Question

Pourquoi ne faut-il pas inclure `Participant`, `File`, `Window`, `Level` ou `Label` dans les features du modèle ?

### Réponse 

...

In [ ]:
# Préparation des données d’entraînement

print("Nombre d'exemples :", ...)
print("Nombre de features :", ...)
print("Exemples de features :", ..)

## 8. Classification EEG — premiers modèles

On teste plusieurs modèles classiques :

- LDA ;
- SVM ;
- Random Forest ;
- KNN ;
- Naive Bayes ;
- Decision Tree ;
- AdaBoost ;
- MLP.

La normalisation `StandardScaler` est placée dans le `sklearn.pipeline.Pipeline` pour éviter une fuite de données entre apprentissage et test. Il faut ajuster le `StandardScaler` uniquement sur les données d’entraînement :

`scaler.fit_transform(X_train)`

Puis appliquer la transformation aux données de test avec :

`scaler.transform(X_test)`

## 9. Évaluation par validation croisée et par sujet

Deux évaluations sont demandées :

### 10-fold cross-validation

Les segments sont répartis en 10 folds stratifiés. Cette évaluation est utile pour comparer les modèles, mais elle peut mélanger les sujets entre apprentissage et test.

### Leave-One-Subject-Out, LOSO

Un sujet est laissé de côté pour le test, tandis que le modèle est entraîné sur les autres sujets. Cette stratégie d’évaluation est plus réaliste, car elle permet de tester la capacité de généralisation du modèle sur un conducteur jamais vu auparavant. L’opération est ensuite répétée sur l’ensemble des sujets disponibles afin d’obtenir une évaluation plus robuste.

## Question

Pourquoi le LOSO est-il souvent plus difficile que le 10-fold classique ?

### Réponse 

...

## 10. Interprétation et discussion

Répondez aux questions suivantes dans le notebook :

1. Quel modèle obtient le meilleur F1-score en 10-fold ?
2. Quel modèle obtient le meilleur F1-score en LOSO ?
3. Les performances chutent-elles en LOSO ? Pourquoi ?
4. Les classes sont-elles équilibrées ?
5. Les résultats obtenus avec EEG seul vous semblent-ils suffisants pour une application réelle ?
6. Quelles limites voyez-vous à l'utilisation des labels subjectifs PAAS ?
7. Quelles améliorations proposeriez-vous ?

## 11. Mini-système d'adaptation

À partir de la prédiction du modèle, on peut simuler une décision d'adaptation.

Exemple :

| Prédiction | Décision |
|---|---|
| charge faible | interface normale |
| charge élevée | simplification de l'interface |
| charge élevée persistante | alerte conducteur |

## Question 

Pourquoi faut-il être prudent avant de déclencher une alerte sur une seule prédiction ?

### Réponse 

...

In [ ]:
def decision_system(...):
    """
    Transforme les prédictions en décision d'adaptation.
    
    """




## 12. Extension optionnelle — vers la multimodalité

Le cœur du TP est volontairement limité à l'EEG.

Une extension possible consiste à reproduire les mêmes étapes pour les autres modalités :

```text
ECG_Features_10s → Normalized_Features_10s/ECG → Normalized_Features_10s_With_Label/ECG
EDA_Features_10s → Normalized_Features_10s/EDA → Normalized_Features_10s_With_Label/EDA
Gaze_Features_10s → Normalized_Features_10s/Gaze → Normalized_Features_10s_With_Label/Gaze
```

Puis à fusionner les features :

```text
EEG + ECG
EEG + EDA
EEG + Gaze
EEG + ECG + EDA + Gaze
```

La fusion la plus simple est une concaténation des colonnes de features pour des fenêtres correspondant au même sujet, au même niveau et au même indice de fenêtre.

## Question

Pourquoi la multimodalité peut-elle améliorer la détection de la charge cognitive ?

### Réponse 

...